In [2]:
import pandas as pd

df = pd.read_csv(r"C:\Users\arunt\Downloads\INDIA_AQI_COMPLETE_20251126.csv")

print(df.columns)

Index(['City', 'State', 'Latitude', 'Longitude', 'Datetime', 'Year', 'Month',
       'Day', 'Hour', 'Day_of_Week', 'Day_Name', 'Week_of_Year', 'Is_Weekend',
       'Quarter', 'Season', 'Time_of_Day', 'Temp_2m_C', 'Temp_80m_C',
       'Temp_120m_C', 'Temp_180m_C', 'Humidity_Percent', 'Dew_Point_C',
       'Humidity_Category', 'Wind_Speed_10m_kmh', 'Wind_Speed_80m_kmh',
       'Wind_Speed_120m_kmh', 'Wind_Dir_10m', 'Wind_Gusts_kmh',
       'Wind_Category', 'Wind_Stagnation', 'Precipitation_mm', 'Rain_mm',
       'Is_Raining', 'Heavy_Rain', 'Pressure_MSL_hPa', 'Surface_Pressure_hPa',
       'Solar_Radiation_Wm2', 'Direct_Radiation_Wm2', 'Diffuse_Radiation_Wm2',
       'UV_Index', 'Cloud_Cover_Percent', 'Cloud_Low_Percent',
       'Cloud_Mid_Percent', 'Cloud_High_Percent', 'Is_Daytime',
       'Sunshine_Seconds', 'PM2_5_ugm3', 'PM10_ugm3', 'PM_Ratio', 'CO_ugm3',
       'NO2_ugm3', 'SO2_ugm3', 'O3_ugm3', 'Dust_ugm3', 'NH3_ugm3', 'AOD',
       'US_AQI', 'US_AQI_PM25', 'US_AQI_PM10', 'US_AQI_

In [2]:
# =========================================================
# FINAL OPTIMIZED AQI FORECAST SYSTEM
# FAST + ERROR FREE + INDUSTRY LEVEL
# =========================================================

# =========================
# IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import joblib

warnings.filterwarnings("ignore")

# =========================
# MACHINE LEARNING
# =========================

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LinearRegression

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

# =========================================================
# CREATE FOLDERS
# =========================================================

os.makedirs("models", exist_ok=True)

os.makedirs("visualizations", exist_ok=True)

# =========================================================
# LOAD DATASET
# =========================================================

print("=" * 70)

print("LOADING AQI DATASET...")

print("=" * 70)

file_path = r"C:\Users\arunt\Downloads\INDIA_AQI_COMPLETE_20251126.csv"

df = pd.read_csv(file_path)

# =========================================================
# FAST SAMPLING
# =========================================================

if len(df) > 50000:

    df = df.sample(

        50000,

        random_state=42

    )

print("\nDATASET LOADED SUCCESSFULLY!")

print("\nUSING DATASET SIZE:", df.shape)

# =========================================================
# REMOVE DUPLICATES
# =========================================================

df.drop_duplicates(inplace=True)

# =========================================================
# HANDLE MISSING VALUES
# =========================================================

df.fillna(method="ffill", inplace=True)

df.fillna(method="bfill", inplace=True)

# =========================================================
# TARGET COLUMN
# =========================================================

target_column = "US_AQI"

# =========================================================
# IMPORTANT FEATURES
# =========================================================

selected_features = [

    "PM2_5_ugm3",
    "PM10_ugm3",
    "CO_ugm3",
    "NO2_ugm3",
    "SO2_ugm3",
    "O3_ugm3",
    "Temp_2m_C",
    "Humidity_Percent",
    "Wind_Speed_10m_kmh",
    "Pressure_MSL_hPa",
    "Cloud_Cover_Percent"

]

# =========================================================
# CHECK FEATURES
# =========================================================

missing_features = []

for feature in selected_features:

    if feature not in df.columns:

        missing_features.append(feature)

if len(missing_features) > 0:

    print("\nMISSING FEATURES:")

    print(missing_features)

    exit()

# =========================================================
# FEATURES & TARGET
# =========================================================

X = df[selected_features]

y = df[target_column]

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42

)

# =========================================================
# OPTIMIZED MODELS
# =========================================================

models = {

    "Linear Regression":

        LinearRegression(),

    "Decision Tree":

        DecisionTreeRegressor(
            max_depth=10
        ),

    "Random Forest":

        RandomForestRegressor(

            n_estimators=30,

            max_depth=12,

            random_state=42,

            n_jobs=-1

        ),

    "Gradient Boosting":

        GradientBoostingRegressor(

            n_estimators=50,

            max_depth=5

        )

}

# =========================================================
# TRAIN MODELS
# =========================================================

results = []

best_model = None

best_score = -999

best_model_name = ""

for name, model in models.items():

    print("\n" + "=" * 70)

    print(f"TRAINING MODEL: {name}")

    print("=" * 70)

    # TRAIN

    model.fit(X_train, y_train)

    # PREDICT

    predictions = model.predict(X_test)

    # METRICS

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    mse = mean_squared_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_test,
        predictions
    )

    print(f"\nMAE : {round(mae, 2)}")

    print(f"RMSE: {round(rmse, 2)}")

    print(f"R² SCORE: {round(r2, 4)}")

    results.append({

        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2

    })

    # BEST MODEL

    if r2 > best_score:

        best_score = r2

        best_model = model

        best_model_name = name

# =========================================================
# SAVE BEST MODEL
# =========================================================

joblib.dump(

    best_model,

    "models/best_aqi_model.pkl"

)

print("\nBEST MODEL SAVED SUCCESSFULLY!")

print(f"\nBEST MODEL: {best_model_name}")

# =========================================================
# RESULTS DATAFRAME
# =========================================================

results_df = pd.DataFrame(results)

print("\nMODEL COMPARISON:\n")

print(results_df)

# =========================================================
# VISUALIZATION 1
# MODEL COMPARISON
# =========================================================

plt.figure(figsize=(10, 5))

sns.barplot(

    data=results_df,

    x="Model",

    y="R2 Score"

)

plt.title("Machine Learning Model Comparison")

plt.tight_layout()

plt.savefig(
    "visualizations/model_comparison.png"
)

plt.close()

# =========================================================
# VISUALIZATION 2
# FAST HEATMAP
# =========================================================

plt.figure(figsize=(12, 8))

corr_data = df[selected_features].corr()

sns.heatmap(

    corr_data,

    cmap="coolwarm"

)

plt.title("Feature Correlation Heatmap")

plt.tight_layout()

plt.savefig(
    "visualizations/heatmap.png"
)

plt.close()

# =========================================================
# VISUALIZATION 3
# AQI DISTRIBUTION
# =========================================================

plt.figure(figsize=(10, 5))

sns.histplot(

    y,

    bins=40,

    kde=True

)

plt.title("AQI Distribution")

plt.xlabel("US AQI")

plt.ylabel("Frequency")

plt.tight_layout()

plt.savefig(
    "visualizations/aqi_distribution.png"
)

plt.close()

# =========================================================
# VISUALIZATION 4
# FEATURE IMPORTANCE
# =========================================================

if hasattr(best_model, "feature_importances_"):

    importance = pd.DataFrame({

        "Feature": selected_features,

        "Importance":
            best_model.feature_importances_

    })

    importance = importance.sort_values(

        by="Importance",

        ascending=False

    )

    plt.figure(figsize=(10, 6))

    sns.barplot(

        data=importance,

        x="Importance",

        y="Feature"

    )

    plt.title("Feature Importance")

    plt.tight_layout()

    plt.savefig(
        "visualizations/feature_importance.png"
    )

    plt.close()

# =========================================================
# VISUALIZATION 5
# ACTUAL VS PREDICTED
# =========================================================

predictions = best_model.predict(X_test)

plt.figure(figsize=(10, 5))

plt.plot(
    y_test.values[:50],
    label="Actual AQI"
)

plt.plot(
    predictions[:50],
    label="Predicted AQI"
)

plt.legend()

plt.title("Actual vs Predicted AQI")

plt.tight_layout()

plt.savefig(
    "visualizations/prediction_vs_actual.png"
)

plt.close()

# =========================================================
# SAMPLE PREDICTION
# =========================================================

sample = X_test.iloc[0:1]

prediction = best_model.predict(sample)

print("\nSAMPLE AQI PREDICTION:")

print(round(prediction[0], 2))

print("\nALL VISUALIZATIONS GENERATED!")

print("\nPROJECT COMPLETED SUCCESSFULLY!")

LOADING AQI DATASET...

DATASET LOADED SUCCESSFULLY!

USING DATASET SIZE: (50000, 71)

TRAINING MODEL: Linear Regression

MAE : 19.1
RMSE: 37.04
R² SCORE: 0.7258

TRAINING MODEL: Decision Tree

MAE : 15.73
RMSE: 35.96
R² SCORE: 0.7416

TRAINING MODEL: Random Forest

MAE : 14.41
RMSE: 35.03
R² SCORE: 0.7547

TRAINING MODEL: Gradient Boosting

MAE : 14.66
RMSE: 33.02
R² SCORE: 0.7821

BEST MODEL SAVED SUCCESSFULLY!

BEST MODEL: Gradient Boosting

MODEL COMPARISON:

               Model        MAE       RMSE  R2 Score
0  Linear Regression  19.103866  37.042105  0.725792
1      Decision Tree  15.728152  35.960230  0.741576
2      Random Forest  14.406676  35.033744  0.754720
3  Gradient Boosting  14.658035  33.022310  0.782077

SAMPLE AQI PREDICTION:
42.65

ALL VISUALIZATIONS GENERATED!

PROJECT COMPLETED SUCCESSFULLY!
